In [ ]:
import numpy as np
import pandas as pd
# from nltk import bigrams,trigrams
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from textblob import TextBlob
from sklearn.decomposition import TruncatedSVD
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
from nltk.stem import PorterStemmer
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download('wordnet') # first-time use only
import nltk, string, numpy
nltk.download('punkt') # first-time use only
import sys
np.set_printoptions(threshold=sys.maxsize)
from sklearn import preprocessing
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [ ]:
#Read review Content File

In [ ]:
file1 = open("/content/reviewContent")
review = defaultdict(list)
for row in file1:
    # temp = row.split()
    text = re.split(r'\t+', row)
    row = re.sub('[^a-zA-Z0-9 \n\..]', '', text[-1])
    review[text[1]].append(row)

file1.close()

In [ ]:
#Read Feature CSV

In [ ]:
file1 = open("/content/featuresAll.csv")
arr = []
for row in file1:
    text = re.split(r'\t+', row)
    arr.append(text)

file1.close()

In [ ]:
#print(arr[1])

In [ ]:
#a1 = preprocessing.normalize(np.array(arr[1:]), norm='max', axis=1)

In [ ]:
#print(review['1'])

In [ ]:
#print(data1[0])

In [ ]:
#Tokenize the reviews to get content similarty matrix

In [ ]:

stemmer = nltk.stem.porter.PorterStemmer()
def StemTokens(tokens):
    return [stemmer.stem(token) for token in tokens]
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
def StemNormalize(text):
    return StemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))
lemmer = nltk.stem.WordNetLemmatizer()
def LemTokens(tokens):
    return [lemmer.lemmatize(token) for token in tokens]
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
def LemNormalize(text):
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))
TfidfVec = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english')
def cos_similarity(textlist):
    tfidf = TfidfVec.fit_transform(textlist)
    return (tfidf * tfidf.T).toarray()
sim = defaultdict(list)
for val in review:
    sim[val].append(cos_similarity(review[val]))

sim_final = defaultdict(list)
for val in sim:
  # for i in range(len(sim[val][0])):
  sim_final[val].append(numpy.mean(sim[val][0], axis = 1))
  sim_final[val].append(numpy.var(sim[val][0], axis = 1))

normalized_sim = defaultdict(list)
for val in sim_final:
  normalized_sim[val].append(preprocessing.normalize(np.array(sim_final[val]), norm='max', axis=1))

b = []
for val in normalized_sim:
  for i in range(len(normalized_sim[val][0][0])):
    b.append([val, normalized_sim[val][0][0][i], normalized_sim[val][0][1][i]])

ar = np.array(b)
np.savetxt('norm_similarity.csv', ar, delimiter=',',fmt='%s',header="ProductID,MeanSimilarity,Variance")

In [ ]:
#print(sim_final['3'])
#print(a[1])

In [ ]:
#Read Review Graph file

In [ ]:
file2 = open("/content/reviewGraph")
dic = defaultdict(list)
for row in file2:
    tr = row.split()
    dic[tr[1]].append(float(tr[-1]))
file2.close()

In [ ]:
#Read Metadata file

In [ ]:
file2 = open("/content/metadata")
d1 = []
d2 = []
for row in file2:
    tr = row.split()
    if int(tr[3]) == -1:
      d1.append(row)
    else:
      d2.append(row)
file2.close()

In [ ]:
#Create Product related features

In [ ]:

ar = []
for val in dic:
  ar.append([val, sum(dic[val])/len(dic[val]), 1 if len(dic[val]) == 1 else 0 , len(dic[val])])

a = np.array(ar)
np.savetxt('product.csv', a, delimiter=',',fmt='%s',header="ProductID,MeanRating,Signleton,numberOfReviews")
print(a)

In [ ]:
# print(len(d1))
# print(len(d2))

In [ ]:
#Train models and feature importance

In [ ]:
df = pd.read_csv('/content/featuresAll.csv')
labels = pd.read_csv('/content/labels.csv')

In [ ]:
from numpy import mean
from numpy import std
from pandas import read_csv
from matplotlib import pyplot
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import BaggingClassifier

# load the dataset
def load_dataset(full_path):
	# load the dataset as a numpy array
	dataframe = read_csv(full_path, header=None, na_values='?')
	# drop rows with missing
	dataframe = dataframe.dropna()
	# split into inputs and outputs
	last_ix = len(dataframe.columns) - 1
	X, y = dataframe.drop(last_ix, axis=1), dataframe[last_ix]
	# select categorical and numerical features
	cat_ix = X.select_dtypes(include=['object', 'bool']).columns
	num_ix = X.select_dtypes(include=['int64', 'float64']).columns
	# label encode the target variable to have the classes 0 and 1
	y = LabelEncoder().fit_transform(y)
	return X.values, y, cat_ix, num_ix

# evaluate a model
def evaluate_model(X, y, model):
  # define evaluation procedure
  cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
  #evaluate model
  #scores = cross_val_score(model, X, y, scoring=['f1-score'], cv=cv, n_jobs=-1)

  scores = cross_validate(model, X, y, scoring=['precision', 'recall', 'f1', 'accuracy'], cv=cv, n_jobs=-1 )
  return scores

# define models to test
def get_models():
	models, names = list(), list()
	# CART
	#models.append(DecisionTreeClassifier())
	#names.append('CART')
	# SVM
	models.append(SVC(gamma='scale'))
	names.append('SVM')
	# Bagging
	#models.append(BaggingClassifier(n_estimators=100))
	#names.append('BAG')
	# RF
	#models.append(RandomForestClassifier(n_estimators=100))
	#names.append('RF')
	# GBM
	#models.append(GradientBoostingClassifier(n_estimators=100))
	#names.append('GBM')
	return models, names

# define the location of the dataset
#full_path = 'adult-all.csv'
# load the dataset
#X, y, cat_ix, num_ix = load_dataset(full_path)
X = df
Y = labels
# define models
models, names = get_models()
results = list()
# evaluate each model
for i in range(len(models)):
  # define steps
  #steps = [('c',OneHotEncoder(handle_unknown='ignore'),cat_ix), ('n',MinMaxScaler(),num_ix)]
  # one hot encode categorical, normalize numerical
  #ct = ColumnTransformer(steps)
  # wrap the model i a pipeline
  pipeline = Pipeline(steps=[('m',models[i])])
  # evaluate the model and store results
  scores = evaluate_model(X, Y, pipeline)
  print("SCORES")
  print(scores)
  results.append(scores)
  # summarize performance
  #print('>%s %.3f (%.3f)' % (names[i], mean(scores), std(scores)))
# plot the results
pyplot.boxplot(results, labels=names, showmeans=True)
pyplot.show()

In [ ]:
X1 = df.to_numpy()
Y = labels.to_numpy()
print(Y.shape)

In [ ]:
model = RandomForestClassifier(n_estimators=100)

pca = PCA(n_components=7)
skf = StratifiedKFold(n_splits=10)
fold_no = 1
for train_index, test_index in skf.split(X1, Y):
    X1_train = X1[train_index,:]
    X1_test = X1[test_index,:]
    Y_train = Y[train_index,:]
    Y_test = Y[test_index,:]
    pca.fit(X1_train)
    X_train = pca.transform(X1_train)
    X_test = pca.transform(X1_test)  
    model.fit(X_train, np.ravel(Y_train,order='C'))
    predictions = model.predict(X_test)
    print('Fold',str(fold_no),'Accuracy:',accuracy_score(Y_test,np.round(predictions)), ' Precision: ',precision_score(Y_test,np.round(predictions)), ' Recall: ',recall_score(Y_test,np.round(predictions)), ' F1-Score: ',f1_score(Y_test,np.round(predictions)))
    fold_no += 1

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
model1 = RandomForestClassifier(n_estimators=100)
A = df.to_numpy()
b = labels.to_numpy()
skf = StratifiedKFold(n_splits=5)
skf.get_n_splits(A, b)

for train_ix, test_ix in skf.split(A, b):
  train_X, test_X = A[train_ix], A[test_ix]
  train_y, test_y = b[train_ix], b[test_ix]
  train_0, train_1 = len(train_y[train_y==0]), len(train_y[train_y==1])
  test_0, test_1 = len(test_y[test_y==0]), len(test_y[test_y==1])
  print('>Train: 0=%d, 1=%d, Test: 0=%d, 1=%d' % (train_0, train_1, test_0, test_1))

  model1.fit(train_X, np.ravel(train_y,order='C'))
  y_pred = model1.predict(test_X)
  print(confusion_matrix(test_y,y_pred))
  print(classification_report(test_y,y_pred))
  print(accuracy_score(test_y, y_pred))

In [ ]:
importances = model1.feature_importances_
std = np.std([tree.feature_importances_ for tree in model1.estimators_],
             axis=0)
indices = np.argsort(importances)[::-1]

In [ ]:
indices

In [ ]:
importances